# From one neuron to a language model

Run the cells from top to bottom in a Python kernel. All the data is defined here. Panel 1 uses plain Python, and panels 2 and 3 use PyTorch on the CPU. If your kernel has no PyTorch, run `%pip install torch` in a new cell first.

Each panel follows the same loop **predict → measure the loss → compute gradients → update parameters**.

| Panel | What the model sees | What it predicts |
| --- | --- | --- |
| 1. Linear neuron | One number | One number |
| 2. Bigram model | One character | The next character |
| 3. Causal Transformer | A window of characters | The next character at every position |

The last model is a small, GPT-style model trained from random weights on a synthetic corpus. Its small vocabulary and short sentences make it possible to inspect what it learns on a CPU.

## 1. Train one neuron by hand

The training points come from $y = 2x + 1$. We start with $w = b = 0$ and predict $\hat y = wx+b$. For each point, the error is $e=\hat y-y$. The mean of $e^2$ is the loss.

The derivative of $e^2$ with respect to the prediction is $2e$. The prediction changes by $x$ per unit of $w$ and by $1$ per unit of $b$. By the chain rule, the gradients are the means of $2ex$ and $2e$. We subtract a small multiple of each gradient to reduce the loss.

In [ ]:
training_points = [(-2, -3), (-1, -1), (0, 1), (1, 3), (2, 5)]
weight, bias = 0.0, 0.0
learning_rate = 0.1

for step in range(101):
    squared_error = 0.0
    weight_gradient = 0.0
    bias_gradient = 0.0

    for x, target in training_points:
        prediction = weight * x + bias
        error = prediction - target
        squared_error += error ** 2
        weight_gradient += 2 * error * x
        bias_gradient += 2 * error

    n_points = len(training_points)
    loss = squared_error / n_points
    if step in (0, 5, 20, 100):
        print(f"step {step:3d} | loss {loss:9.6f} | w {weight:.4f} | b {bias:.4f}")

    weight -= learning_rate * weight_gradient / n_points
    bias -= learning_rate * bias_gradient / n_points

x = 3
print(f"Held-out input {x} → prediction {weight * x + bias:.4f} (target 7)")

## 2. Predict the next character from one character

Each distinct character gets an integer token ID. A window of text is the input, and the same window shifted one character to the right supplies the targets. In `red` → `ed `, the target for `r` is `e`, the target for `e` is `d`, and the target for `d` is a space.

The synthetic corpus ties **red** to **stop** and **green** to **go**. Both words follow the same suffix, `light means `. This creates a case where a one-character model sees identical inputs but a model with context can distinguish them. We reserve some adjective–place combinations for validation before forming any windows. Each individual adjective and place still occurs in training.

In [ ]:
from random import Random

import torch
from torch import nn
from torch.nn import functional as F

# Small matrix operations run faster here with a single CPU thread.
torch.set_num_threads(1)
context_length = 24
batch_size = 32

adjectives = ("old", "new", "quiet", "busy")
places = ("bridge", "station", "square", "market")
signals = {"red": "stop", "green": "go"}
train_lines, validation_lines = [], []

for adjective_index, adjective in enumerate(adjectives):
    for place_index, place in enumerate(places):
        lines = validation_lines if adjective_index == place_index else train_lines
        for colour, action in signals.items():
            lines.append(f"at the {adjective} {place}, a {colour} light means {action}.\n")

shuffler = Random(42)
shuffler.shuffle(train_lines)
shuffler.shuffle(validation_lines)
train_text = "".join(train_lines)
validation_text = "".join(validation_lines)

vocabulary = sorted(set(train_text))
token_id = {character: index for index, character in enumerate(vocabulary)}


def encode(text):
    return torch.tensor([token_id[character] for character in text], dtype=torch.long)


def decode(ids):
    return "".join(vocabulary[index] for index in ids)


train_ids = encode(train_text)
validation_ids = encode(validation_text)


def sample_batch(tokens, generator):
    starts = torch.randint(len(tokens) - context_length, (batch_size,), generator=generator)
    positions = starts[:, None] + torch.arange(context_length)
    return tokens[positions], tokens[positions + 1]


print(f"{len(train_lines)} training lines, {len(validation_lines)} validation lines")
print(f"Vocabulary has {len(vocabulary)} characters")
print(f"Input  {decode(train_ids[:context_length])!r}")
print(f"Target {decode(train_ids[1:context_length + 1])!r}")

The bigram model is a learnable table with one row per input character and one column per possible next character. Looking up a row produces the same result as multiplying a one-hot input by a weight matrix. Its outputs are **logits**, or raw scores. Cross-entropy measures the negative log probability assigned to each correct next character and averages over the batch.

`loss.backward()` computes gradients by the chain rule, which we applied by hand in panel 1. `optimizer.step()` applies the update. We use Adam here to adapt the update size for each parameter, and clear gradients before each batch because PyTorch accumulates them.

In [ ]:
class BigramModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.next_token_scores = nn.Embedding(vocab_size, vocab_size)
        nn.init.zeros_(self.next_token_scores.weight)

    def forward(self, tokens):
        return self.next_token_scores(tokens)


def next_token_loss(model, inputs, targets):
    # The model returns [batch, time, vocabulary]. Score each position.
    logits = model(inputs)
    return F.cross_entropy(logits.flatten(0, 1), targets.flatten())


@torch.no_grad()
def evaluate(model, tokens):
    model.eval()
    # Use the same validation windows each time, independent of training.
    generator = torch.Generator().manual_seed(123)
    losses = []
    for _ in range(8):
        inputs, targets = sample_batch(tokens, generator)
        losses.append(next_token_loss(model, inputs, targets).item())
    return sum(losses) / len(losses)


def train_language_model(model, steps, learning_rate):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    generator = torch.Generator().manual_seed(42)
    print(f"step {0:3d} | validation loss {evaluate(model, validation_ids):.3f}")

    model.train()
    for step in range(1, steps + 1):
        inputs, targets = sample_batch(train_ids, generator)
        optimizer.zero_grad()
        loss = next_token_loss(model, inputs, targets)  # Predict and measure.
        loss.backward()                                # Compute gradients.
        optimizer.step()                               # Update parameters.

        if step % max(50, steps // 5) == 0 or step == steps:
            validation_loss = evaluate(model, validation_ids)
            print(f"step {step:3d} | batch loss {loss.item():.3f} | validation loss {validation_loss:.3f}")
            model.train()

    model.eval()


bigram = BigramModel(len(vocabulary))
train_language_model(bigram, steps=150, learning_rate=0.1)

## 3. Give the model a memory of earlier characters

The Transformer keeps the same data, loss, and training loop. It starts with a learned vector for each token and adds a learned vector for its position in the window.

- **Self-attention** lets each position combine information from other positions. A query represents what a position is looking for, keys are what positions match against, and values are the information they share. The scaled query–key dot products become weights through softmax.
- **The causal mask** sets scores for future positions to $-\infty$ before softmax, giving them zero weight. Each position can therefore predict its target without seeing it.
- **The feed-forward layer** transforms each position independently through two linear layers and a GELU activation. Residual connections add each transformation back to its input, and layer normalization keeps the scale of activations controlled. The final linear layer converts each position's vector into vocabulary logits.

This implementation uses one attention head and one Transformer block. [Attention Is All You Need](https://arxiv.org/abs/1706.03762)

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, width, max_context):
        super().__init__()
        self.query = nn.Linear(width, width)
        self.key = nn.Linear(width, width)
        self.value = nn.Linear(width, width)
        self.output = nn.Linear(width, width)
        self.scale = width ** -0.5
        self.register_buffer(
            "future_positions",
            torch.ones(max_context, max_context, dtype=torch.bool).triu(diagonal=1),
        )

    def forward(self, features):
        time = features.size(1)
        queries = self.query(features)
        keys = self.key(features)
        values = self.value(features)
        scores = (queries @ keys.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(self.future_positions[:time, :time], float("-inf"))
        weights = scores.softmax(dim=-1)
        return self.output(weights @ values)


class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, max_context, width=32):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, width)
        self.position_embedding = nn.Embedding(max_context, width)
        self.attention_norm = nn.LayerNorm(width)
        self.attention = CausalSelfAttention(width, max_context)
        self.feed_forward_norm = nn.LayerNorm(width)
        self.feed_forward = nn.Sequential(
            nn.Linear(width, 4 * width),
            nn.GELU(),
            nn.Linear(4 * width, width),
        )
        self.output_norm = nn.LayerNorm(width)
        self.vocabulary_head = nn.Linear(width, vocab_size)

    def forward(self, tokens):
        positions = torch.arange(tokens.size(1), device=tokens.device)
        features = self.token_embedding(tokens) + self.position_embedding(positions)
        features = features + self.attention(self.attention_norm(features))
        features = features + self.feed_forward(self.feed_forward_norm(features))
        return self.vocabulary_head(self.output_norm(features))


torch.manual_seed(42)
transformer = TinyTransformer(len(vocabulary), context_length)
parameter_count = sum(parameter.numel() for parameter in transformer.parameters())
print(f"{parameter_count:,} trainable parameters")
train_language_model(transformer, steps=1000, learning_rate=0.003)

### Check what context buys us

Both prompts come from held-out combinations and end in the same space character. Compare the probabilities for `s` and `g`, the first characters of the two possible actions. The bigram model must give the same probabilities for both prompts. The Transformer can use the colour within its 24-character window.

In [ ]:
held_out_place = f"{adjectives[0]} {places[0]}"
prompts = [f"at the {held_out_place}, a {colour} light means " for colour in signals]


@torch.no_grad()
def next_character_probabilities(model, prompt, temperature=1.0):
    tokens = encode(prompt[-context_length:]).unsqueeze(0)
    last_logits = model(tokens)[0, -1]
    return (last_logits / temperature).softmax(dim=-1)


for colour, prompt in zip(signals, prompts):
    print(f"Prompt {prompt!r} (expected {signals[colour]!r})")
    for name, model in (("bigram", bigram), ("Transformer", transformer)):
        probabilities = next_character_probabilities(model, prompt)
        print(f"  {name:11s} | P('s') {probabilities[token_id['s']]:.3f}"
              f" | P('g') {probabilities[token_id['g']]:.3f}")

### Generate a continuation

Sample one character, append it to the prompt, and run the model again on the newest context window. Temperature scales the logits before sampling. A lower temperature concentrates more probability on the highest-scoring characters. Try changing the prompt using characters from the training vocabulary, or rerun panel 3 with fewer training steps and compare the probabilities.

In [ ]:
def generate(model, prompt, max_new_characters=12, temperature=0.7, seed=7):
    generator = torch.Generator().manual_seed(seed)
    text = prompt
    for _ in range(max_new_characters):
        probabilities = next_character_probabilities(model, text, temperature)
        next_id = torch.multinomial(probabilities, num_samples=1, generator=generator).item()
        next_character = vocabulary[next_id]
        text += next_character
        if next_character == "\n":
            break
    return text


for prompt in prompts:
    print(generate(transformer, prompt).rstrip())